### Data

Sample data: 
one line per userL user id \t [[x,y,z],[...],...] (train items)[[a,b,c],[...]...] [test items] \t [[...]](fixed negatives)

Each item is [poi_id, time_slot, neg_id]: neg_id is no longer used, these are drawn every epoch
- timeslot is only used in TLDA

poi2_xy: one line per POI: poid_id \t lon \t lat
tlda_model_file: 

TLDA: 
- docuemnt: one user
- word: one check-in: POI + time slot: the time slot is what make the TLDA temporal
- topic: lifesyle pattern (commute metro station in the morininngs, or go to chinese restaurant at lunch)

main_tlda only reads column one, the training items, for each user it builds a list of (user,poi, time_slot)

instead of storing dictionaries from IDs to rows, it uses IDs directly as array positions, so it finds the largest user id, poi id and time slots and adds 1. 

If our vocab has say 10,000 words, a topic is sumply a vector os 10,000 numbers that sum to 1.0
e.g: Topic 1: (football); [ball:0.05, astronomy: 0.001,...]

For every single word in the datset, we assingme a random topic ID. 

To make LDA work we only need to track down two matrices: 
- doc_topic_counts: D x K: counts how many words document d has assingmed to topic k
- topic_word_counts: KxV: counts words v to topci k cooccurence

Gibbs sampling: for an specific word token (poi,time slot)
- unassign temporarily: substract from count matrices this words current topic: 
e.g: docu_topic_counts[user i,topic k] -= 1
time_topic[time_slot, topic k] -= 1
topic_word_counts[topic k, poi id] -= 1
topic_word_counts[topic k] -= 1 ()  
- Calculate probaility for every topic: calculate how likley this word is to belong to each topic k
    word-topic-score - doc-topic-score

word topic score; across the whole data, how much does this word associate with topic k

doc topic score: within thsi document, hwo dominant is the topic k among the other words

- Sample new topic: rolls a weighted die to picj new topic form the probaility distribution and update the counts matrices

Repeat on k iterations. The paper does not specify stopping criterion, so we will do one long run, see where it flaterns and set iter accordingly. 
    




In [1]:
from construct_category_path import CHECKIN_FILES, PROCESSED_DIR, load_checkins
from collections import Counter,defaultdict
from statistics import mean, median
import json
import numpy as np
import pandas as pd

# One pipeline for every city: each step is a function, and the last cell runs all steps
# for NYC and Tokyo. Both cities are processed identically, which RQ2 (cross-city) requires.


def build_venue_table(data):
    """
    Median lat/lon per venue, categoryId, n_checkins, n_users
    """

    raw_venues = defaultdict(lambda: {"coords": [], "categories": [], "users": []})

    for row in data.itertuples(index=False):
        entry = raw_venues[row.venueId]
        entry["coords"].append((row.lat, row.lon))
        entry["categories"].append(row.categoryId)
        entry["users"].append(row.userId)

    venue_table = {}
    for venue, info in raw_venues.items():
        med_lat, med_lon = np.median(info["coords"], axis=0)
        # Pick the most frequent categoryId for the venue
        top_category = Counter(info["categories"]).most_common(1)[0][0]

        venue_table[venue] = {
            "lat": float(med_lat),
            "lon": float(med_lon),
            "categoryId": top_category,
            "n_checkins": len(info["coords"]),
            "n_users": len(set(info["users"])),
        }


    return venue_table

In [2]:
def first_visits(data, min_pois=10):
    """
     each user's distinct POIs in order of first visit.

    sort check-ins by userId, utc_time, venueId (utc_time = absolute moment; venueId breaks ties)
    drop duplicate (userId, venueId) and keep the first = the first visit, with its slot
    keep users with n >= min_pois distinct POIs; this drops only one user for NYC

    Returns {userId: [(venueId, first utc_time, slot of the first visit), ...]} in time order.
    """
    data = data.sort_values(['userId', 'utc_time', 'venueId'])
    data = data.drop_duplicates(subset=['userId', 'venueId'], keep='first')

    user_venues_visited = defaultdict(list)
    for row in data.itertuples():
        user_venues_visited[row.userId].append((row.venueId, row.utc_time, row.slot))

    # rows were sorted before the loop, so every list is already in time order
    return {user: visits for user, visits in user_venues_visited.items() if len(visits) >= min_pois}


def describe_histories(city, users_venue_history):
    """Number of users and distinct POIs per user."""
    lengths = [len(visits) for visits in users_venue_history.values()]
    print(f"{city}: Users: {len(lengths)}, Avg: {mean(lengths):.2f}, Min: {min(lengths)}, "
          f"Max: {max(lengths)}, Median: {median(lengths)}")

In [3]:
def split_user_venues_train_val_test(user_venue_history, train_frac=0.70, val_frac=0.15):

    """
    Step 7: split each user's POIs by position: first 70% train, next 15% val, rest test.

    floor(x + 0.5) instead of round(): numpy rounds .5 to the nearest even number.
    Each POI has exactly one position, so train, val and test never overlap.
    Returns {userId: {'train': [...], 'val': [...], 'test': [...]}}, same tuples as the input.
    """

    user_train_val_test_splits = {}
    for user, visits in user_venue_history.items():
        n = len(visits)
        n_train = int(np.floor(train_frac * n + 0.5))
        n_val = int(np.floor(val_frac * n + 0.5))
        user_train_val_test_splits[user] = {
            'train': visits[:n_train],
            'val': visits[n_train:n_train + n_val],
            'test': visits[n_train + n_val:],
        }
    return user_train_val_test_splits

In [4]:
def training_checkins(data, user_splits):
    """
    Step 8: the check-ins TLDA and TEMN may learn from.

    cutoff = utc_time of the user's first validation POI.
    Keeps every check-in of a kept user before the cutoff, repeats included (TLDA needs them).
    Everything after the cutoff, also repeat visits to train POIs, is not used anywhere.
    """
    cutoff = pd.Series({user: s['val'][0][1] for user, s in user_splits.items()}, name='cutoff')
    data = data.join(cutoff, on='userId', how='inner')  # inner: drops the filtered-out users
    train = data[data['utc_time'] < data['cutoff']]
    return train.sort_values(['userId', 'utc_time']).drop(columns='cutoff').reset_index(drop=True)

In [5]:
def renumber_and_save(city, user_splits, train_checkins, venue_table):
    """
    Step 9: renumber IDs (strings to integers) and save the maps and the split table to
    data/processed/<city>/. Returns the tables the later steps need.
    """
    # users 0..U-1 in sorted order of the raw userId
    user_ids = sorted(user_splits)
    user_map = pd.DataFrame({'user_id': range(len(user_ids)), 'userId': user_ids})

    # POIs in two blocks: training POIs 0..V_train-1 first (TLDA sizes its tables from the largest
    # training POI id), then every other venue of the city (held-out only, or never trained on)
    train_venues = sorted(train_checkins['venueId'].unique())
    other_venues = sorted(set(venue_table) - set(train_venues))
    poi_map = pd.DataFrame({'venueId': train_venues + other_venues})
    poi_map['poi_id'] = range(len(poi_map))
    poi_map['in_train'] = poi_map['poi_id'] < len(train_venues)
    venues_df = pd.DataFrame.from_dict(venue_table, orient='index')[['categoryId', 'lat', 'lon']]
    poi_map = poi_map.join(venues_df, on='venueId')[['poi_id', 'venueId', 'categoryId', 'lat', 'lon', 'in_train']]

    user_to_id = dict(zip(user_map['userId'], user_map['user_id']))
    venue_to_id = dict(zip(poi_map['venueId'], poi_map['poi_id']))

    # one row per (user, POI): its position, split, first-visit time and slot (needed later for RQ1)
    rows = []
    for user, s in user_splits.items():
        for split_name in ('train', 'val', 'test'):
            for venue, first_time, slot in s[split_name]:
                rows.append((user, venue, split_name, first_time, slot))
    split_table = pd.DataFrame(rows, columns=['userId', 'venueId', 'split', 'first_utc_time', 'slot'])
    split_table['position'] = split_table.groupby('userId').cumcount()
    split_table['user_id'] = split_table['userId'].map(user_to_id)
    split_table['poi_id'] = split_table['venueId'].map(venue_to_id)

    out_dir = PROCESSED_DIR / city.lower()
    out_dir.mkdir(parents=True, exist_ok=True)
    user_map.to_csv(out_dir / 'user_map.csv', index=False)
    poi_map.to_csv(out_dir / 'poi_map.csv', index=False)
    split_table.to_parquet(out_dir / 'split.parquet', index=False)
    print(f"{city}: users: {len(user_map)}, training POIs: {len(train_venues)}, "
          f"all POIs: {len(poi_map)}, saved to {out_dir}")
    return dict(city=city, out_dir=out_dir, user_map=user_map, poi_map=poi_map,
                split_table=split_table, user_to_id=user_to_id, venue_to_id=venue_to_id,
                n_train_venues=len(train_venues))

In [6]:
def check_part_a(tables, train_checkins):
    """Part C checks that concern Part A."""
    split_table, venue_to_id = tables['split_table'], tables['venue_to_id']
    sets = split_table.groupby(['userId', 'split'])['venueId'].agg(set).unstack()
    # 1. per user, train / val / test POIs do not overlap
    for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
        assert all(not (x & y) for x, y in zip(sets[a], sets[b])), f"{a} and {b} overlap"
    # 2. per user: last train first-visit < first val first-visit < first test first-visit
    t = split_table.groupby(['userId', 'split'])['first_utc_time'].agg(['min', 'max']).unstack()
    assert (t[('max', 'train')] < t[('min', 'val')]).all() and (t[('max', 'val')] < t[('min', 'test')]).all()
    # 3. every training check-in is at one of that user's train POIs
    merged = train_checkins.merge(split_table[['userId', 'venueId', 'split']], on=['userId', 'venueId'], how='left')
    assert (merged['split'] == 'train').all()
    # 4. each (user, POI) appears once, i.e. every held-out item is a first visit
    assert not split_table.duplicated(['userId', 'venueId']).any()
    # 5. ids without gaps, slots within 0..47
    assert sorted(tables['user_map']['user_id']) == list(range(len(tables['user_map'])))
    assert set(train_checkins['venueId'].map(venue_to_id)) == set(range(tables['n_train_venues']))
    assert train_checkins['slot'].between(0, 47).all() and train_checkins['slot'].nunique() == 48
    print(f"{tables['city']}: all Part A checks passed")

### Part B: write the files the model reads

`data_<city>_val` / `data_<city>_test` (e.g. `data_nyc_val`): one line per user, `user_id \t col1 \t col2 \t col4`
- col1: every training check-in as `[poi_id, slot, 0]`, in time order, repeats included (identical in both files)
- col2: the held-out POIs as `[poi_id, slot of the first visit, 0]`: val POIs or test POIs
- col4: POIs excluded from the negatives: `[]` in the val file, the user's val POIs in the test file

`poi2_xy`: `poi_id \t lon \t lat` for the training POIs only (longitude first)

In [7]:
def write_model_files(tables, train_checkins):
    """Steps 10-11: data_<city>_val, data_<city>_test and poi2_xy in data/processed/<city>/."""
    city, out_dir = tables['city'].lower(), tables['out_dir']
    user_map, poi_map, split_table = tables['user_map'], tables['poi_map'], tables['split_table']
    user_to_id, venue_to_id = tables['user_to_id'], tables['venue_to_id']

    # Step 10. column 1: the training check-ins per user, already sorted by userId, utc_time
    train_items = defaultdict(list)
    for row in train_checkins.itertuples():
        # int(): json.dumps cannot write numpy integers
        train_items[user_to_id[row.userId]].append([int(venue_to_id[row.venueId]), int(row.slot), 0])

    # column 2: the held-out POIs per user, in order of discovery
    heldout = {'val': defaultdict(list), 'test': defaultdict(list)}
    for row in split_table[split_table['split'] != 'train'].sort_values(['user_id', 'position']).itertuples():
        heldout[row.split][int(row.user_id)].append([int(row.poi_id), int(row.slot), 0])

    def write_data_file(file_path, heldout_items, excluded_pois):
        """One line per user, users in order 0..U-1."""
        with open(file_path, 'w') as f:
            for user_id in range(len(user_map)):
                columns = [train_items[user_id], heldout_items[user_id], excluded_pois(user_id)]
                f.write(f"{user_id}\t" + "\t".join(json.dumps(c) for c in columns) + "\n")

    write_data_file(out_dir / f'data_{city}_val', heldout['val'], lambda u: [])
    # in the test file the user's val POIs are visited places, so they must never be drawn as negatives
    write_data_file(out_dir / f'data_{city}_test', heldout['test'], lambda u: [item[0] for item in heldout['val'][u]])

    # Step 11. poi2_xy: training POIs only, longitude first.
    # float(): the repr of a numpy float is 'np.float64(...)', not a number
    with open(out_dir / 'poi2_xy', 'w') as f:
        for row in poi_map[poi_map['in_train']].itertuples():
            f.write(f"{row.poi_id}\t{float(row.lon)!r}\t{float(row.lat)!r}\n")

In [8]:
def check_model_files(tables):
    """Part C checks 6-7: read the files back the way the model reads them."""
    city, out_dir, split_table = tables['city'].lower(), tables['out_dir'], tables['split_table']
    poi2xy_ids = {int(line.split('\t')[0]) for line in open(out_dir / 'poi2_xy')}
    assert poi2xy_ids == set(range(tables['n_train_venues']))

    for split_name in ('val', 'test'):
        name = f'data_{city}_{split_name}'
        lines = open(out_dir / name).read().splitlines()
        assert len(lines) == len(tables['user_map'])
        n_heldout = 0
        for expected_user, line in enumerate(lines):
            user_id, col1, col2, col4 = line.split('\t')
            col1, col2, col4 = json.loads(col1), json.loads(col2), json.loads(col4)
            assert int(user_id) == expected_user                       # users in order 0..U-1
            train_pois = {item[0] for item in col1}
            heldout_pois = {item[0] for item in col2}
            assert train_pois <= poi2xy_ids                            # 7. column 1 only uses trained POIs
            assert not heldout_pois & train_pois                       # 6. held-out POIs are new to the user
            assert not heldout_pois & set(col4)                        # 6. and not excluded by mistake
            assert all(0 <= item[1] <= 47 for item in col1 + col2)
            n_heldout += len(col2)
        assert n_heldout == (split_table['split'] == split_name).sum()
        print(f"{name}: {len(lines)} users, {n_heldout} held-out pairs: checks passed")

### Run the pipeline for each city

Steps 1-11 plus all checks, once per city. Outputs go to `data/processed/<city>/`.
A city whose raw file is missing from `data/raw/` is skipped with a message.

In [9]:
def preprocess_city(city):
    """Steps 1-11 and all checks for one city ('NYC' or 'TKY')."""
    # Steps 1-3: load (local time + slot are added by load_checkins), drop duplicate check-ins
    # (same user, venue and moment; utc_time, not local_time, identifies the moment)
    data = load_checkins(city)
    data = data.drop_duplicates(subset=['userId', 'venueId', 'utc_time']).reset_index(drop=True)
    venue_table = build_venue_table(data)                                # step 4

    users_venue_history = first_visits(data)                             # steps 5-6
    describe_histories(city, users_venue_history)

    user_splits = split_user_venues_train_val_test(users_venue_history)  # step 7
    assert all(len(s['val']) >= 1 and len(s['test']) >= 1 for s in user_splits.values())
    print({name: sum(len(s[name]) for s in user_splits.values()) for name in ('train', 'val', 'test')})

    train_checkins = training_checkins(data, user_splits)                # step 8
    print(len(train_checkins), "training check-ins,", train_checkins['venueId'].nunique(), "training POIs")

    tables = renumber_and_save(city, user_splits, train_checkins, venue_table)  # step 9
    check_part_a(tables, train_checkins)
    write_model_files(tables, train_checkins)                           # steps 10-11
    check_model_files(tables)
    return tables


results = {}
for city in ['NYC', 'TKY']:
    if not CHECKIN_FILES[city].exists():
        print(f"\n{city}: skipped, {CHECKIN_FILES[city]} not found")
        continue
    print(f"\n===== {city}")
    results[city] = preprocess_city(city)


===== NYC
NYC: Users: 1082, Avg: 84.12, Min: 10, Max: 714, Median: 76.0
{'train': 63751, 'val': 13693, 'test': 13571}
141944 training check-ins, 30358 training POIs
NYC: users: 1082, training POIs: 30358, all POIs: 38333, saved to C:\Users\Ruben\Downloads\TEMN-py313\TEMN\data\processed\nyc
NYC: all Part A checks passed
data_nyc_val: 1082 users, 13693 held-out pairs: checks passed
data_nyc_test: 1082 users, 13571 held-out pairs: checks passed

===== TKY
TKY: Users: 2290, Avg: 92.55, Min: 10, Max: 904, Median: 80.0
{'train': 148415, 'val': 31863, 'test': 31660}
360536 training check-ins, 48917 training POIs
TKY: users: 2290, training POIs: 48917, all POIs: 61858, saved to C:\Users\Ruben\Downloads\TEMN-py313\TEMN\data\processed\tky
TKY: all Part A checks passed
data_tky_val: 2290 users, 31863 held-out pairs: checks passed
data_tky_test: 2290 users, 31660 held-out pairs: checks passed
